# Day 33 — CI/CD and eval-gated release

Your RAG service is a container image (Day 32). How does a change to it — new code, a tweaked
prompt, a re-chunked index — reach production **without a regression reaching users first**?

A normal CI pipeline runs lint + unit tests. For an LLM system that is not enough: a prompt
edit can pass every unit test and still make answers worse. So the pipeline gets an extra
gate — the **eval suite** from Week 9 — and the rollout gets **progressive delivery** (canary,
shadow, automatic rollback).

Everything here is built from scratch and runs offline with a fake judge.

## Learning objectives

1. Model a CI pipeline as ordered stages with fail-fast and artifacts passed between them.
2. Build an **eval gate**: run the suite, compare to a stored baseline, block on regression.
3. Read a GitHub Actions workflow and map each job to a pipeline stage.
4. Explain blue/green vs canary vs shadow deployment and what each protects against.
5. Implement a canary controller: traffic split → metric collection → SLO check → promote/rollback.
6. Explain why prompt + code + index must be versioned and rolled back **together**.

## Agenda (60 min)

| # | Segment | Time |
| - | ------- | ---- |
| 0 | The question: safe path from commit to prod | 3 min |
| 1 | "push to main = deploy", and the regression it ships | 8 min |
| 2 | A pipeline runner: stages, fail-fast, artifacts | 12 min |
| 3 | The eval gate: baseline, threshold, allow-list | 13 min |
| 4 | The same pipeline in GitHub Actions | 6 min |
| 5 | Progressive delivery: canary with auto-rollback | 15 min |
| 6 | Versioning the whole bundle; bridge to monitoring | 3 min |
| 7 | Exercises and self-check quiz | — |

## Setup

```bash
source ../../../.venv/bin/activate
```

Pure standard library + a fixed random seed. No API calls.


## 0 — The question (3 min)

Continuous Integration = every change is automatically built and checked. Continuous
Delivery/Deployment = every change that passes is automatically releasable / released.

For an LLM app the pipeline has **four kinds of check**, in increasing cost and value:

| Stage | Catches | Cost |
| ----- | ------- | ---- |
| lint / type | syntax, obvious bugs | ms |
| unit tests | broken logic, contracts | seconds |
| **eval suite** | **quality regressions** (worse answers, more hallucination) | minutes + $ |
| canary in prod | what only real traffic reveals (latency, edge inputs) | minutes of exposure |

Days 25–27 built the eval suite. Today it becomes a **merge gate**.

## 1 — "push to main = deploy", and the regression it ships (8 min)

In [1]:
import random
random.seed(0)

# a tiny "quality" model: score in [0,1] for how good an answer is, given a prompt version
def answer_quality(prompt_version, case):
    base = {"v1": 0.82, "v2-terse": 0.60}[prompt_version]   # v2 "reads cleaner" but grounds worse
    return max(0.0, min(1.0, random.gauss(base, 0.08)))

def unit_tests(prompt_version):
    # checks the shape: non-empty, <= 400 words, cites a source. Both versions pass.
    return {"non_empty": True, "under_limit": True, "has_citation": True}

CASES = [f"q{i}" for i in range(40)]

for pv in ("v1", "v2-terse"):
    passed = all(unit_tests(pv).values())
    mean_q = sum(answer_quality(pv, c) for c in CASES) / len(CASES)
    print(f"{pv:10s}  unit tests pass={passed}   mean answer quality={mean_q:.3f}")
print("\n-> v2-terse ships: green pipeline, 0.20 drop in grounded quality, users find out")


v1          unit tests pass=True   mean answer quality=0.811
v2-terse    unit tests pass=True   mean answer quality=0.597

-> v2-terse ships: green pipeline, 0.20 drop in grounded quality, users find out


Unit tests check *structure* (non-empty, cited, within limits). They cannot see that the
answers got vaguer. Only an eval that *scores content* — against references or with a judge —
catches this. So add that stage, and make a regression fail the build.

## 2 — A pipeline runner: stages, fail-fast, artifacts (12 min)

A pipeline is an ordered list of stages. Each stage gets a shared `context` dict (artifacts
from earlier stages), returns updates, and may fail — which stops the pipeline (fail-fast).

In [2]:
from dataclasses import dataclass, field
import time as _t

@dataclass
class StageResult:
    name: str; ok: bool; detail: str; artifacts: dict = field(default_factory=dict)

class Pipeline:
    def __init__(self, stages): self.stages = stages    # list of (name, fn)
    def run(self, context):
        log = []
        for name, fn in self.stages:
            try:
                res = fn(context)
            except Exception as e:
                res = StageResult(name, False, f"{type(e).__name__}: {e}")
            log.append(res)
            print(f"  [{'ok ' if res.ok else 'FAIL'}] {name:14s} {res.detail}")
            if not res.ok:
                print(f"  pipeline stopped at '{name}'")
                return False, log
            context.update(res.artifacts)
        print("  pipeline passed")
        return True, log

def stage_lint(ctx):
    return StageResult("lint", True, "0 issues")

def stage_unit(ctx):
    ok = all(unit_tests(ctx["prompt_version"]).values())
    return StageResult("unit", ok, "12 passed" if ok else "failures")

def stage_build(ctx):
    tag = f"rag-api:{ctx['git_sha']}"
    return StageResult("build", True, f"image {tag}", {"image_tag": tag})

pipe = Pipeline([("lint", stage_lint), ("unit", stage_unit), ("build", stage_build)])
ok, _ = pipe.run({"prompt_version": "v1", "git_sha": "abc123"})


  [ok ] lint           0 issues
  [ok ] unit           12 passed
  [ok ] build          image rag-api:abc123
  pipeline passed


## 3 — The eval gate: baseline, threshold, allow-list (13 min)

The gate: run the eval suite on the candidate, compare the aggregate score **and** per-case
scores to a stored **baseline** (the last thing that shipped). Block if:

- aggregate drops by more than `tolerance`, **or**
- any case regresses from pass→fail that isn't on an explicit **allow-list** (a change you
  intended — e.g. you rewrote that case).

In [3]:
import json, statistics

def run_eval_suite(prompt_version):
    """Returns {case: score}. In real life: llmlab.judge() over your eval set."""
    return {c: answer_quality(prompt_version, c) for c in CASES}

def eval_gate(candidate_scores, baseline, *, tolerance=0.03, pass_mark=0.7, allow_regress=()):
    cand_mean = statistics.mean(candidate_scores.values())
    base_mean = statistics.mean(baseline.values())
    delta = cand_mean - base_mean

    regressions = []
    for c, s in candidate_scores.items():
        was_pass = baseline.get(c, 0) >= pass_mark
        now_pass = s >= pass_mark
        if was_pass and not now_pass and c not in allow_regress:
            regressions.append(c)

    ok = (delta >= -tolerance) and not regressions
    return {
        "ok": ok,
        "baseline_mean": round(base_mean, 3),
        "candidate_mean": round(cand_mean, 3),
        "delta": round(delta, 3),
        "unexpected_regressions": regressions[:5],
        "n_regressions": len(regressions),
    }

# freeze a baseline from the currently-shipping prompt:
baseline = run_eval_suite("v1")

print("candidate = v1 (no change):")
print(json.dumps(eval_gate(run_eval_suite("v1"), baseline), indent=2))
print("\ncandidate = v2-terse:")
print(json.dumps(eval_gate(run_eval_suite("v2-terse"), baseline), indent=2))


candidate = v1 (no change):
{
  "ok": false,
  "baseline_mean": 0.817,
  "candidate_mean": 0.831,
  "delta": 0.013,
  "unexpected_regressions": [
    "q2",
    "q11",
    "q27",
    "q31"
  ],
  "n_regressions": 4
}

candidate = v2-terse:
{
  "ok": false,
  "baseline_mean": 0.817,
  "candidate_mean": 0.589,
  "delta": -0.228,
  "unexpected_regressions": [
    "q0",
    "q1",
    "q2",
    "q3",
    "q4"
  ],
  "n_regressions": 37
}


In [4]:
# wire it into the pipeline as a stage
def stage_eval(ctx):
    baseline = ctx["baseline_scores"]
    scores = run_eval_suite(ctx["prompt_version"])
    verdict = eval_gate(scores, baseline, allow_regress=ctx.get("allow_regress", ()))
    detail = f"mean {verdict['candidate_mean']} vs {verdict['baseline_mean']} (Δ{verdict['delta']:+}), {verdict['n_regressions']} regressions"
    return StageResult("eval-gate", verdict["ok"], detail, {"eval_verdict": verdict})

full = Pipeline([("lint", stage_lint), ("unit", stage_unit),
                 ("eval-gate", stage_eval), ("build", stage_build)])

print("--- candidate v2-terse ---")
full.run({"prompt_version": "v2-terse", "git_sha": "def456", "baseline_scores": baseline})
print("\n--- candidate v1 ---")
full.run({"prompt_version": "v1", "git_sha": "aaa111", "baseline_scores": baseline})


--- candidate v2-terse ---
  [ok ] lint           0 issues
  [ok ] unit           12 passed
  [FAIL] eval-gate      mean 0.6 vs 0.817 (Δ-0.217), 34 regressions
  pipeline stopped at 'eval-gate'

--- candidate v1 ---
  [ok ] lint           0 issues
  [ok ] unit           12 passed
  [FAIL] eval-gate      mean 0.821 vs 0.817 (Δ+0.004), 2 regressions
  pipeline stopped at 'eval-gate'


(False,
 [StageResult(name='lint', ok=True, detail='0 issues', artifacts={}),
  StageResult(name='unit', ok=True, detail='12 passed', artifacts={}),
  StageResult(name='eval-gate', ok=False, detail='mean 0.821 vs 0.817 (Δ+0.004), 2 regressions', artifacts={'eval_verdict': {'ok': False, 'baseline_mean': 0.817, 'candidate_mean': 0.821, 'delta': 0.004, 'unexpected_regressions': ['q1', 'q17'], 'n_regressions': 2}})])

Notes that matter in practice:

- **The baseline is an artifact**, versioned next to the code (or in object storage keyed by
  the deployed SHA). When a candidate ships, its scores *become* the new baseline.
- **Eval cost is real** — judge calls cost money and minutes. Run the full suite on
  `main`/release; run a fast subset on every PR.
- **Flakiness** — LLM scores vary run to run. Use `tolerance`, average over N runs for the
  aggregate, and require regressions to reproduce before blocking.
- **The allow-list is a deliberate escape hatch**, reviewed in the PR — not a way to silence
  the gate.

## 4 — The same pipeline in GitHub Actions (6 min)

In [5]:
workflow = '''name: ci
on:
  pull_request:
  push: { branches: [main] }

jobs:
  check:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.12" }
      - run: pip install -r requirements.txt
      - run: ruff check .                       # lint stage
      - run: pytest -q -m "not live"            # unit stage
      - name: eval gate                         # the LLM-specific stage
        env: { ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }} }
        run: |
          python -m evals.run --suite evals/regression.jsonl \
            --baseline artifacts/baseline_$(git rev-parse origin/main).json \
            --tolerance 0.03 --fast ${{ github.event_name == 'pull_request' }}

  release:
    needs: check
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    environment: production          # requires manual approval + branch protection
    steps:
      - uses: actions/checkout@v4
      - run: ./scripts/build_and_push.sh $GITHUB_SHA        # build stage (Day 32 image)
      - run: terraform apply -auto-approve -var image_tag=$GITHUB_SHA
      - run: python -m deploy.canary --tag $GITHUB_SHA      # progressive rollout (§5)
'''
from pathlib import Path
Path("ci.yml").write_text(workflow)
print(workflow)


name: ci
on:
  pull_request:
  push: { branches: [main] }

jobs:
  check:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.12" }
      - run: pip install -r requirements.txt
      - run: ruff check .                       # lint stage
      - run: pytest -q -m "not live"            # unit stage
      - name: eval gate                         # the LLM-specific stage
        env: { ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }} }
        run: |
          python -m evals.run --suite evals/regression.jsonl             --baseline artifacts/baseline_$(git rev-parse origin/main).json             --tolerance 0.03 --fast ${{ github.event_name == 'pull_request' }}

  release:
    needs: check
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    environment: production          # requires manual approval + branch protection
    steps:
      - uses: actions/checkout@v4
      - ru

`jobs` map 1:1 to our stages; `needs: check` is the ordering; `environment: production` is
the approval gate; **branch protection** makes `check` a *required* status so nothing merges
red. Secrets come from `secrets.*`, never from the repo.

## 5 — Progressive delivery: canary with auto-rollback (15 min)

Even a green pipeline can't see everything (real latency under load, weird inputs, a provider
hiccup). So don't cut 100% of traffic at once.

| Strategy | How | Protects against |
| -------- | --- | ---------------- |
| **Blue/green** | stand up v2 fully, flip the LB, keep v1 warm | fast rollback; but 100% exposure at the flip |
| **Canary** | send X% to v2, watch metrics, ramp 5→25→50→100 | limits blast radius; auto-rollback on breach |
| **Shadow** | mirror real requests to v2, **don't return** its responses | zero user risk; compare quality/latency offline |

Build a canary controller: it holds two versions, splits traffic, collects metrics per
version, checks them against SLOs, and ramps or rolls back.

In [6]:
import random, statistics
random.seed(7)

def serve(version, case):
    """Simulate one request. v2 is a bit slower and has a higher error rate."""
    profile = {"v1": dict(lat=0.9, err=0.010, q=0.82),
               "v2": dict(lat=1.2, err=0.015, q=0.83)}[version]  # v2: slightly slower, still within SLO
    errored = random.random() < profile["err"]
    latency = max(0.05, random.gauss(profile["lat"], 0.25))
    quality = 0.0 if errored else max(0, min(1, random.gauss(profile["q"], 0.08)))
    return {"error": errored, "latency_s": latency, "quality": quality}

SLO = {"p95_latency_s": 2.0, "error_rate": 0.03, "min_quality": 0.75}

class Canary:
    def __init__(self, stable, candidate, steps=(5, 25, 50, 100)):
        self.stable, self.candidate, self.steps = stable, candidate, list(steps)
        self.pct = 0

    def _metrics(self, version, n=400):
        rs = [serve(version, i) for i in range(n)]
        lat = sorted(r["latency_s"] for r in rs)
        return {"p95_latency_s": round(lat[int(0.95 * n)], 2),
                "error_rate": round(sum(r["error"] for r in rs) / n, 3),
                "quality": round(statistics.mean(r["quality"] for r in rs), 3)}

    def _healthy(self, m):
        return (m["p95_latency_s"] <= SLO["p95_latency_s"]
                and m["error_rate"] <= SLO["error_rate"]
                and m["quality"] >= SLO["min_quality"])

    def roll_out(self):
        for step in self.steps:
            self.pct = step
            m = self._metrics(self.candidate)
            status = "healthy" if self._healthy(m) else "SLO BREACH"
            print(f"  {self.pct:3d}% -> {self.candidate}  {m}  [{status}]")
            if not self._healthy(m):
                self.pct = 0
                print(f"  ROLLBACK: all traffic back to {self.stable}")
                return "rolled_back"
        print(f"  PROMOTED: {self.candidate} is now stable at 100%")
        return "promoted"

print("Canary: v1 (stable) -> v2 (candidate)")
Canary("v1", "v2").roll_out()


Canary: v1 (stable) -> v2 (candidate)
    5% -> v2  {'p95_latency_s': 1.63, 'error_rate': 0.018, 'quality': 0.814}  [healthy]
   25% -> v2  {'p95_latency_s': 1.63, 'error_rate': 0.013, 'quality': 0.818}  [healthy]
   50% -> v2  {'p95_latency_s': 1.61, 'error_rate': 0.02, 'quality': 0.81}  [healthy]
  100% -> v2  {'p95_latency_s': 1.62, 'error_rate': 0.013, 'quality': 0.821}  [healthy]
  PROMOTED: v2 is now stable at 100%


'promoted'

In [7]:
# make v2 clearly bad and watch it get caught at the first step
def serve_bad(version, case):
    if version == "v2":
        return {"error": random.random() < 0.18, "latency_s": max(0.05, random.gauss(2.6, 0.4)),
                "quality": max(0, min(1, random.gauss(0.7, 0.1)))}
    return serve("v1", case)

_real_serve = serve
serve = serve_bad
try:
    Canary("v1", "v2").roll_out()
finally:
    serve = _real_serve


    5% -> v2  {'p95_latency_s': 3.35, 'error_rate': 0.2, 'quality': 0.701}  [SLO BREACH]
  ROLLBACK: all traffic back to v1


The controller needs three things you must design deliberately:

1. **A guardrail metric set** — latency p95, error rate, *and* a quality proxy (judge score on
   a sample, or a cheap heuristic like citation-present rate). Latency alone misses "fast and
   wrong."
2. **Enough exposure to be significant** — 5% for 30 seconds is noise. Ramp on time *and*
   request count; for small deltas you need sequential testing, not a single peek.
3. **A definition of "roll back"** — see §6.

## 6 — Versioning the whole bundle; bridge to monitoring (3 min)

A RAG answer is a function of **code × prompt × retrieval index × model version**. If you roll
back the code but not the index (which was re-chunked in the same release), you get a
combination that was never tested. So:

- Ship them as **one versioned bundle** — `release-2026-09-10.4` pins the image tag, the
  prompt file hash, the index URI (`s3://kb/v7`), and the model id.
- Rollback = redeploy the previous bundle id, atomically.
- The eval baseline is keyed by bundle id too.

**Where this goes next:** Day 34 — once v2 is at 100%, monitoring takes over: golden signals,
drift detection on the inputs, and the loop that decides when the system needs a refresh.

## Exercises

1. **Fast vs full suite.** Add a `--fast` mode to `run_eval_suite` that samples 8 of the 40
   cases (seeded). Run the gate in fast mode on 20 "PRs" (random prompt tweaks) and count how
   often fast disagrees with full. What tolerance makes fast safe as a PR gate?
2. **Flake handling.** Make `answer_quality` noisier (σ=0.15). Modify `eval_gate` to average
   the aggregate over N=5 runs and to require a per-case regression to appear in ≥3/5 runs
   before blocking. Show it stops blocking on noise but still catches `v2-terse`.
3. **Shadow deployment.** Implement `Shadow(stable, candidate)`: for each request, serve
   `stable` to the user, also call `candidate`, log `(latency, quality)` for both, return only
   `stable`. After 500 requests print a comparison table. When would you promote?
4. **Ramp on significance.** Replace the fixed `steps` with a rule: stay at each percentage
   until you've seen ≥300 candidate requests *and* the error-rate confidence interval
   excludes the SLO. Use a normal approximation for the CI.
5. **Bundle rollback.** Model a `Release` object (`image_tag`, `prompt_hash`, `index_uri`,
   `model`). Write `deploy(release)` and `rollback()` that always restore a *complete*
   previous bundle. Show that rolling back code alone would produce an untested combo.
6. **GitHub Actions dissection.** For the `ci.yml` in §4, mark which steps run on a PR vs on
   `main`, where secrets enter, and what `branch protection` + `environment: production` each
   enforce. What happens if `check` is *not* a required status?

## Self-check quiz

1. Why are lint + unit tests insufficient as the only gate for a prompt change? Give a
   concrete failure they'd miss.
2. What three inputs does the eval gate compare, and what are the two block conditions?
3. What is the eval **baseline**, where does it live, and when does it get updated?
4. Blue/green vs canary vs shadow — one sentence each on what it protects against.
5. Name the three metric families a canary controller must watch and why latency alone is
   not enough.
6. Why must code, prompt, and index be rolled back together? Give an example of the bug you
   get if you don't.
7. In GitHub Actions, what makes a failing `check` job actually *prevent* a merge?
